# **Implicit Classifier**

In this colab, you will implement ways to aggregate the joint states of the tactile dataset, and train a classifier that outputs contact (binary), slope, and material using just the joint states and no tactile data at all!

Let's first mount the colab to your Google Drive.

> **Run record.** These outputs come from a local run on an Apple M4 laptop (PyTorch MPS backend), 2026-11-02, 77 min, status: ok. The printed paths show the local copy of the dataset; in Colab `data_dir` is `/content/drive/MyDrive/tactile_data`.

In [1]:
# We will only save the processed dataset to your Drive to save disk space
from google.colab import drive
drive.mount('/content/drive')

[local run] drive.mount('/content/drive') skipped; data_dir points at a local copy


Load in all related libraries, and the directory of the tactile dataset. Once again, replace `data_dir` with your own Google Drive directory of the dataset:

In [2]:
import os
import torch
import pickle
import random
import copy
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from scipy.interpolate import interp1d
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader, random_split

data_dir = '/content/drive/MyDrive/tactile_data'

random.seed(0)
np.random.seed(0)
torch.manual_seed(0);

/private/tmp/claude-501/-Users-tejkosaraju-Downloads-classes-cs123/15934c31-ca98-4461-a1c3-e3b9024ec06a/scratchpad/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataset

Now we begin processing the dataset. Although the collected data isn't perfectly formatted, the `WindowedJointStateDataset` class handles all inconsistencies in the dataset structure.

It loads the joint state data — a 9-dimensional vector containing the angular **position**, **velocity**, and **effort** (the handout said "acceleration"; the CSV columns are `*_effort`) of each motor on Pupper's right leg. The dataset slides over this data using a fixed stride of `window_stride` frames, and captures a history of `window_size` frames at each step. It interpolates along nearest image captures to determine whether Pupper's leg was in contact in a certain instance. Unlike the `DenseTactDataset`, we are **not** eliminating no-contact instances for this dataset.

>**Reminder:** The ROS 2 system from Lab 3 records data at **1000 frames per second**, so a `window_size` of 100 corresponds to a 0.1-second observation window.

In [3]:
class WindowedJointStateDataset(Dataset):
    def __init__(self, root_dir, window_size, window_stride):
        self.root_dir = root_dir
        self.window_size = window_size
        self.window_stride = window_stride
        self.incline_classes = [15, 30, 45, 60, 75, 0]
        self.material_classes = ['none', 'felt pad', 'rough', 'slip', 'very rough', 'foam']
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.windows = []

        for folder1 in os.listdir(root_dir):
            if folder1.startswith('.') or folder1 in ['reference_images', 'lookups']:
                continue
            for folder2 in os.listdir(os.path.join(root_dir, folder1)):
                if folder2.startswith('.'):
                    continue
                folder_path = os.path.join(root_dir, folder1, folder2)
                if not os.path.isdir(folder_path):
                    continue

                # Parse the path below root_dir so that digits in data_dir cannot set the label.
                incline_label, material_label = self._parse_folder_name(os.path.relpath(folder_path, root_dir))
                if incline_label is None or material_label is None:
                    continue

                print(f"Processing {folder_path}")
                print(f"Incline: {self.incline_classes[incline_label]}, Material: {self.material_classes[material_label]}")
                self._process_folder(folder_path, incline_label, material_label)

    def _parse_folder_name(self, folder_name):
        incline_label = None
        material_label = None
        folder_lower = folder_name.lower()

        for i, incline in enumerate(self.incline_classes):
            if str(incline) in folder_lower:
                incline_label = i
                break

        for i, material in enumerate(self.material_classes):
            if material in folder_lower:
                material_label = i
                break

        if 'felt_pad' in folder_lower or 'felt pad' in folder_lower:
            material_label = self.material_classes.index('felt pad')
        elif 'very_rough' in folder_lower or 'very rough' in folder_lower or 'very-rough' in folder_lower:  # '45-very-rough'
            material_label = self.material_classes.index('very rough')

        return incline_label, material_label

    def _process_folder(self, folder_path, incline_label, material_label):
        js_file = os.path.join(folder_path, 'joint_states.csv')
        ct_file = os.path.join(folder_path, 'contact.csv')
        if not os.path.exists(js_file) or not os.path.exists(ct_file):
            return

        try:
            js = pd.read_csv(js_file).sort_values('timestamp')
            ct = pd.read_csv(ct_file).sort_values('timestamp')
        except Exception as e:
            print(f"Error loading files in {folder_path}: {e}")
            return

        if js.empty or ct.empty or len(js) < self.window_size:
            return

        interp_funcs = [
            interp1d(ct['timestamp'], ct.iloc[:, i+1], bounds_error=False, fill_value='extrapolate')
            for i in range(ct.shape[1] - 1)
        ]

        timestamps = js['timestamp'].values
        motor_states = js.drop(columns='timestamp').iloc[:, :9].values.astype(np.float32)

        contact_interp = np.array([[f(t) for f in interp_funcs] for t in timestamps], dtype=np.float32)
        contact_tensor = torch.tensor(contact_interp, device=self.device)
        contact_states = (contact_tensor.max(dim=1)[0] > 0.5).float().cpu().numpy()

        for i in range(0, len(motor_states) - self.window_size + 1, self.window_stride):
            self.windows.append({
                'motor_states': motor_states[i:i + self.window_size],
                'contact_state': int(np.mean(contact_states[i:i + self.window_size]) > 0.5),
                'contact_sequence': contact_states[i:i + self.window_size],
                'incline_label': incline_label,
                'material_label': material_label
            })

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        return {
            'motor_states': torch.tensor(w['motor_states'], dtype=torch.float32),
            'contact_state': torch.tensor(w['contact_state'], dtype=torch.float32),
            'contact_sequence': torch.tensor(w['contact_sequence'], dtype=torch.float32),
            'incline_label': torch.tensor(w['incline_label'], dtype=torch.long),
            'material_label': torch.tensor(w['material_label'], dtype=torch.long)
        }

### 🔍 Choosing the Optimal `window_size`

Your goal in this part of the lab is to determine the most suitable `window_size` for processing the data.  
We'll keep the `window_stride` fixed at **20** to ensure consistency across experiments.

Once you've selected a `window_size`, run the cell below to initialize the dataset.

> ⚠️ **Note:** Data processing may take a few minutes.  
> For best performance, make sure you're using an **active GPU**.


In [4]:
# window_size = 200 samples = 0.2 s of history at ~1 kHz (see "Choice of window_size" below).
dataset = WindowedJointStateDataset(
    data_dir,
    window_size=200,
    window_stride=20
)
print(f"{len(dataset)} windows")

Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/15 degrees_new/15-very_rough
Incline: 15, Material: very rough


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/15 degrees_new/15-rough
Incline: 15, Material: rough


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/15 degrees_new/15-slip
Incline: 15, Material: slip


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/15 degrees_new/15-felt_pad
Incline: 15, Material: felt pad


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees/45-felt_pad
Incline: 45, Material: felt pad


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees/45-very-rough
Incline: 45, Material: very rough


... (47 outputs left out)

Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees_new/45-felt_pad
Incline: 45, Material: felt pad


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees_new/45-rough
Incline: 45, Material: rough


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees_new/45-slip
Incline: 45, Material: slip


107962 windows


### Saving Processed Dataset to PKL

Once again, processing these joint state into a dataset every single time can be slow and inefficient — we definitely don’t want to repeat that each time we train! To avoid this, we’ll save the filtered dataset into a `.pkl` (pronounced 🥒"pickle") file. In the future, we can simply load this preprocessed dataset directly from the PKL!

We’ve provided the PKL saving script below.

In [5]:
def save_dataset(dataset, filename):
    """Save the processed dataset to avoid reloading next time"""
    save_data = {
        'windows': dataset.windows,
        'incline_classes': dataset.incline_classes,
        'material_classes': dataset.material_classes,
        'window_size': dataset.window_size,
        'window_stride': dataset.window_stride,
    }

    print(f"Saving dataset with {len(dataset.windows)} windows to {filename}")
    with open(filename, 'wb') as f:
        pickle.dump(save_data, f)
    print("Dataset saved successfully!")

# The handout used os.path.join(data_dir, "/lookups/..."): the leading "/" makes
# os.path.join drop data_dir and write to /lookups. One file per window size:
lookup_dir = os.path.join(data_dir, "lookups")
os.makedirs(lookup_dir, exist_ok=True)
dataset_file = os.path.join(lookup_dir, f"processed_dataset_w{dataset.window_size}_s{dataset.window_stride}.pkl")
save_dataset(dataset, dataset_file)

Saving dataset with 107962 windows to /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/lookups/processed_dataset_w200_s20.pkl


Dataset saved successfully!


We provide a script that loads your saved `.pkl` dataset — this is all you need to run after you've selected a `window_size` and saved the dataset to disk.

> 💡 **Tip:** We recommend modifying the save/load filenames so you can store and experiment with multiple datasets using different observation histories (i.e., different `window_size` values).

In [6]:
def load_dataset(filename):
    """Load a previously processed dataset"""
    print(f"Loading dataset from {filename}")
    with open(filename, 'rb') as f:
        save_data = pickle.load(f)

    # Create an empty dataset (no processing)
    dataset = WindowedJointStateDataset.__new__(WindowedJointStateDataset)

    # Restore the saved attributes
    dataset.windows = save_data['windows']
    dataset.incline_classes = save_data['incline_classes']
    dataset.material_classes = save_data['material_classes']
    dataset.window_size = save_data['window_size']
    dataset.window_stride = save_data['window_stride']
    dataset.root_dir = ""  # Not used after loading

    print(f"Loaded dataset with {len(dataset.windows)} windows")
    return dataset

dataset_file = os.path.join(data_dir, "lookups", "processed_dataset_w200_s20.pkl")
dataset = load_dataset(dataset_file)

Loading dataset from /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/lookups/processed_dataset_w200_s20.pkl


Loaded dataset with 107962 windows


### PCGrad

We will again use PCGrad to handle the multi-task learning nature of this setup. In addition to predicting slope incline and surface material, we now also use joint states to predict whether Pupper's leg is in contact with the ground.

This adds a third task to the training objective, which increases the chance of conflicting gradients between tasks. Gradient surgery — the core idea behind PCGrad — is designed to resolve such conflicts, and should (in theory) become even more useful in this setting.

Paste your implementation of `PCGrad` from the Explicit Classifier Notebook into the cell below.


In [7]:
class PCGrad:
    def __init__(self, optimizer, reduction='mean'):
        """
        Args:
            optimizer (torch.optim.Optimizer): Wrapped optimizer.
            reduction (str): Method to reduce gradients ('mean' or 'sum').
        """
        self._optim = optimizer
        self._reduction = reduction

    @property
    def optimizer(self):
        return self._optim

    def zero_grad(self):
        """Clear gradients of all optimized parameters."""
        return self._optim.zero_grad(set_to_none=True)

    def step(self):
        """Performs a single optimization step."""
        return self._optim.step()

    def pc_backward(self, objectives):
        """
        Compute backward pass using Projected Conflicting Gradient (PCGrad).

        Args:
            objectives (List[torch.Tensor]): List of scalar loss values.
        """
        grads, shapes, has_grads = self._pack_grad(objectives)
        pc_grad = self._project_conflicting(grads, has_grads)
        pc_grad = self._unflatten_grad(pc_grad, shapes[0])
        self._set_grad(pc_grad)

    # ---------- Internal Utilities ----------

    def _pack_grad(self, objectives):
        """
        Compute gradients for each objective.

        Returns:
            grads (List[Tensor]): Flattened gradients.
            shapes (List[Tuple]): Shapes of original parameters.
            has_grads (List[Tensor]): Masks indicating valid gradients.
        """
        grads, shapes, has_grads = [], [], []

        for obj in objectives:
            self._optim.zero_grad(set_to_none=True)
            obj.backward(retain_graph=True)
            grad, shape, has_grad = self._retrieve_grad()
            grads.append(self._flatten_grad(grad))
            has_grads.append(self._flatten_grad(has_grad))
            shapes.append(shape)

        return grads, shapes, has_grads

    def _retrieve_grad(self):
        """
        Retrieve gradients and shapes from the model parameters.

        Returns:
            grad (List[Tensor]): Gradient tensors.
            shape (List[Tuple]): Original parameter shapes.
            has_grad (List[Tensor]): Gradient masks.
        """
        grad, shape, has_grad = [], [], []

        for group in self._optim.param_groups:
            for p in group['params']:
                if p.grad is None:
                    shape.append(p.shape)
                    grad.append(torch.zeros_like(p))
                    has_grad.append(torch.zeros_like(p))
                else:
                    shape.append(p.grad.shape)
                    grad.append(p.grad.clone())
                    has_grad.append(torch.ones_like(p))

        return grad, shape, has_grad

    def _flatten_grad(self, grads):
        return torch.cat([g.flatten() for g in grads])

    def _unflatten_grad(self, flat_grads, shapes):
        unflattened, idx = [], 0
        for shape in shapes:
            length = np.prod(shape)
            unflattened.append(flat_grads[idx:idx + length].view(shape).clone())
            idx += length
        return unflattened

    def _set_grad(self, grads):
        """Assign updated gradients back to model parameters."""
        idx = 0
        for group in self._optim.param_groups:
            for p in group['params']:
                p.grad = grads[idx]
                idx += 1

    def _project_conflicting(self, grads, has_grads):
        """
        Resolve gradient conflicts via orthogonal projection.

        Args:
            grads (List[Tensor]): Flattened gradients.
            has_grads (List[Tensor]): Valid gradient masks.

        Returns:
            Tensor: Projected merged gradient.
        """
        shared = torch.stack(has_grads).prod(0).bool()
        pc_grad = copy.deepcopy(grads)
        num_tasks = len(grads)

        for g_i in pc_grad:
            random.shuffle(grads)
            for g_j in grads:
                # Gradient surgery: if g_i conflicts with g_j (negative inner product),
                # remove from g_i its component along g_j (in place, so pc_grad is updated).
                g_i_g_j = torch.dot(g_i, g_j)
                if g_i_g_j < 0:
                    g_i -= g_i_g_j * g_j / (g_j.norm() ** 2)

        merged = torch.zeros_like(grads[0])

        # Handle weights with all contributing gradients
        if self._reduction == 'mean':
            merged[shared] = torch.stack([g[shared] for g in pc_grad]).mean(dim=0)
        elif self._reduction == 'sum':
            merged[shared] = torch.stack([g[shared] for g in pc_grad]).sum(dim=0)
        else:
            raise ValueError("Invalid reduction method")
        # Sum up weights with some non-contributing gradients for stability
        merged[~shared] = torch.stack([g[~shared] for g in pc_grad]).sum(dim=0)
        return merged

### 🧠 JointStateCNN

To process the temporal joint state information, we will use a 1D convolutional neural network. Since each joint state input is a sequence over time (with shape `[window_size, input_channels]`), we apply 1D convolutions **along the time axis** to extract temporal features.

Your task is to complete the implementation of the `JointStateCNN` class:

1. **Add several 1D convolutional layers** that progressively extract higher-level features from the joint state input. Make sure to apply activation functions, batch normalization, and optional pooling layers to reduce the temporal dimension.

2. After the convolutions, **flatten the output** and pass it through a fully connected layer with dropout.

3. Implement **three separate output heads**:
   - `contact_head`: a binary classification head that predicts whether the leg is in contact with the surface.
   - `incline_head`: a multi-class classifier for predicting the slope incline (6 classes).
   - `material_head`: a multi-class classifier for predicting the surface material (6 classes).

> 🔁 Tip: Remember to reshape the input tensor for 1D convolution by transposing it from shape `[batch_size, window_size, input_channels]` to `[batch_size, input_channels, window_size]`.

Use the provided skeleton below to complete your implementation.


In [8]:
class JointStateCNN(torch.nn.Module):
    def __init__(self, input_channels=9, window_size=100):
        super().__init__()

        # 1D Convolutional layers (along time dimension). Each block: conv (kernel 5,
        # "same" padding) -> batch norm -> ReLU -> max-pool by 2. Channels 9 -> 32 -> 64 -> 128.
        self.conv1 = torch.nn.Conv1d(input_channels, 32, kernel_size=5, padding=2)
        self.bn1 = torch.nn.BatchNorm1d(32)
        self.conv2 = torch.nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = torch.nn.BatchNorm1d(64)
        self.conv3 = torch.nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn3 = torch.nn.BatchNorm1d(128)
        self.pool = torch.nn.MaxPool1d(2)

        # Calculate the size after convolutions and pooling
        output_size = window_size
        output_size = output_size // 2  # pool1
        output_size = output_size // 2  # pool2
        output_size = output_size // 2  # pool3

        # Fully connected layers
        self.fc1 = torch.nn.Linear(128 * output_size, 256)
        self.dropout = torch.nn.Dropout(0.5)

        # Output heads: one contact logit (sigmoid in forward), 6 incline logits, 6 material logits
        self.contact_head = torch.nn.Linear(256, 1)
        self.incline_head = torch.nn.Linear(256, 6)
        self.material_head = torch.nn.Linear(256, 6)

    def forward(self, x):
        # x shape: [batch_size, window_size, input_channels]
        # Reshape for 1D convolution: [batch_size, input_channels, window_size]
        x = x.transpose(1, 2)

        # Convolutional layers
        x = self.pool(torch.nn.functional.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.nn.functional.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.nn.functional.relu(self.bn3(self.conv3(x))))

        # Flatten
        x = x.reshape(x.size(0), -1)

        # Fully connected
        x = torch.nn.functional.relu(self.fc1(x))
        x = self.dropout(x)

        # Output heads
        return {
            'contact': torch.sigmoid(self.contact_head(x)),
            'incline': self.incline_head(x),
            'material': self.material_head(x)
        }

### Splitting the Dataset: Train vs. Validation

Now, let’s split the dataset into **90% training** and **10% validation**.

This step is important because the validation set is used to evaluate the model’s performance on **unseen joint state data**. It does not participate in the training process, helping us understand how well the model generalizes to new inputs.

You will help us use a 9:1 split ratio to divide the dataset accordingly, and construct the DataLoader to feed into the model!

As a sanity check, you should see **~90000** train joint states and **~10000** validation joint states after the split, but the exact number of them depends on your observation history.

In [9]:
# Define the split ratio
train_ratio = 0.9
val_ratio = 1 - train_ratio

total_size = len(dataset)
train_size = int(train_ratio * total_size)
val_size = total_size - train_size

train_subset, val_subset = random_split(dataset, [train_size, val_size],
                                        generator=torch.Generator().manual_seed(0))

dataloaders = {
    'train': DataLoader(train_subset, batch_size=512, shuffle=True),
    'val': DataLoader(val_subset, batch_size=512, shuffle=False)
}

print(f"Dataset split into train ({train_size}) and val ({val_size}) successfully!")

Dataset split into train (97165) and val (10797) successfully!


### 🏋️‍♂️ Training the Multi-Task Model

In this section, you'll implement the training loop for our multi-task `JointStateCNN` model. This model is responsible for predicting three things based on joint state sequences:

1. **Contact**: Whether Pupper's leg is in contact with the surface (binary classification).
2. **Incline**: The slope of the surface (6-class classification).
3. **Material**: The type of surface material (6-class classification).

To handle these tasks jointly, we use **PCGrad** to resolve conflicting gradients during backpropagation.

---

### ✂️ Selective Gradient Masking

Because **incline** and **material** predictions are only meaningful when contact is present, we should:
- **Mask out incline and material losses** when the ground-truth contact label is false (i.e., no contact).
- Skip computing gradients for these tasks when there's no contact in the batch. (you will need to implement this part)

This helps avoid penalizing the model for errors on incline/material predictions that aren't physically meaningful when the leg isn't touching the surface.

---

### 📉 Decaying Contact Loss Weight

The **contact prediction task tends to saturate early** in training. To prevent it from dominating gradient updates throughout training, we introduce a **decaying contact loss weight**:
- Start with a weight of 1.0 at epoch 1.
- Linearly decay it to 0.1 by the final epoch.

This allows incline and material predictions to play a larger role in the later stages of training.

---

Once you’ve set up the dataloaders and optimizer, implement the rest of the training loop based on this logic.


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(0)
model = JointStateCNN(input_channels=9, window_size=dataset.window_size).to(device)

bce = torch.nn.BCELoss()
ce = torch.nn.CrossEntropyLoss(reduction='none')

base_optim = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer = PCGrad(base_optim)
num_epochs = 50
history = []

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs}")

    contact_weight = max(0.1, 1.0 - (epoch - 1) / (num_epochs - 1))  # Linear decay

    for phase in ['train', 'val']:
        model.train() if phase == 'train' else model.eval()

        contact_preds, contact_targets = [], []
        incline_preds, incline_targets = [], []
        material_preds, material_targets = [], []

        for batch in tqdm(dataloaders[phase], desc=f"{phase.capitalize()} Epoch {epoch}"):
            x = batch['motor_states'].to(device)
            y_contact = batch['contact_state'].unsqueeze(1).to(device)
            y_incline = batch['incline_label'].to(device)
            y_material = batch['material_label'].to(device)

            if phase == 'train':
                optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                out = model(x)

                loss_contact = bce(out['contact'], y_contact)

                mask = y_contact.bool().float().squeeze()
                loss_incline = ce(out['incline'], y_incline)
                loss_material = ce(out['material'], y_material)

                masked_incline = torch.nan_to_num((loss_incline * mask).mean(), nan=0.0)
                masked_material = torch.nan_to_num((loss_material * mask).mean(), nan=0.0)

                # Decaying contact weight; incline/material objectives only when the
                # batch has contact windows (their masked loss is 0 otherwise).
                weighted_contact = contact_weight * loss_contact
                objectives = [weighted_contact]
                if masked_incline.item() > 0:
                    objectives.append(masked_incline)
                if masked_material.item() > 0:
                    objectives.append(masked_material)

                if phase == 'train':
                    optimizer.pc_backward(objectives)
                    optimizer.step()

            contact_preds.extend((out['contact'] > 0.5).float().cpu().numpy().flatten())
            contact_targets.extend(y_contact.cpu().numpy().flatten())

            contact_indices = mask.cpu().numpy().astype(bool)
            if contact_indices.any():
                incline_preds.extend(torch.argmax(out['incline'], dim=1).cpu().numpy()[contact_indices])
                incline_targets.extend(y_incline.cpu().numpy()[contact_indices])
                material_preds.extend(torch.argmax(out['material'], dim=1).cpu().numpy()[contact_indices])
                material_targets.extend(y_material.cpu().numpy()[contact_indices])

        # Compute and print metrics
        contact_acc = accuracy_score(contact_targets, contact_preds)
        incline_acc = accuracy_score(incline_targets, incline_preds) if incline_targets else 0
        material_acc = accuracy_score(material_targets, material_preds) if material_targets else 0

        history.append((epoch, phase, contact_acc, incline_acc, material_acc))
        print(f"{phase} | Contact Acc: {contact_acc:.4f} | Incline Acc: {incline_acc:.4f} | Material Acc: {material_acc:.4f}")

Using device: mps



Epoch 1/50


train | Contact Acc: 0.9215 | Incline Acc: 0.5266 | Material Acc: 0.3039


val | Contact Acc: 0.9421 | Incline Acc: 0.6187 | Material Acc: 0.4216

Epoch 2/50


train | Contact Acc: 0.9377 | Incline Acc: 0.6203 | Material Acc: 0.3971


val | Contact Acc: 0.9429 | Incline Acc: 0.6893 | Material Acc: 0.4414

Epoch 3/50


... (92 outputs left out)

train | Contact Acc: 0.9531 | Incline Acc: 0.8775 | Material Acc: 0.7955


val | Contact Acc: 0.9580 | Incline Acc: 0.8944 | Material Acc: 0.8190

Epoch 50/50


train | Contact Acc: 0.9536 | Incline Acc: 0.8787 | Material Acc: 0.7977


val | Contact Acc: 0.9592 | Incline Acc: 0.8575 | Material Acc: 0.7903


### Choice of `window_size` (added)

We trained the same model at four history lengths (stride 20) with `analysis/ablation.py`, under two splits:

* **random**: the 90/10 window split of this notebook. Neighbouring windows overlap by (W − 20)/W, so most validation windows have near-duplicates in the training set.
* **blocked**: the last 20 % of every recording is validation, with a one-window gap before it.

Validation accuracy, mean of epochs 16 to 20 (contact / incline / material, %):

| W (samples) | random split | blocked split |
|---|---|---|
| 50 | 94.3 / 79.4 / 68.1 | not run |
| 100 | 94.5 / 79.9 / 69.5 | 94.8 / 75.1 / 63.3 |
| 200 | 94.9 / 82.7 / 73.6 | 94.8 / 79.3 / 68.8 |
| 400 | not run | 95.2 / 81.3 / 69.5 (20-epoch run) |

The 50, 100 and 200 runs have 30 epochs. The 400 run has 20 epochs, so its contact-loss weight decays faster and its row is not fully comparable.

We use **W = 200** (approximately 0.2 s at the dataset's 976 Hz). Contact accuracy changes by less than 1 point with W. Incline and material improve by 3 to 6 points from 100 to 200 samples, and by 2.0 and 0.7 points from 200 to 400. A longer window also delays the window label (the majority vote changes only when half of the window is in contact: 100 ms at W = 200, 200 ms at W = 400) and doubles the size of `fc1`. For a contact detector on the robot, this delay matters.

### Final validation metrics and export for Pupper (added)

The cell below recomputes the final-epoch validation metrics, then writes the trained model
for the deployment challenge:

* `implicit_classifier.pt`: the PyTorch `state_dict`.
* `implicit_classifier.npz`: the same weights as NumPy arrays plus the window size and the
  channel order. Pupper's Raspberry Pi runs this file with NumPy only (`robot/contact_detector.py`),
  so PyTorch is not necessary on the robot.

In [11]:
model.eval()
c_true, c_pred, i_true, i_pred, m_true, m_pred = [], [], [], [], [], []
with torch.no_grad():
    for batch in dataloaders['val']:
        out = model(batch['motor_states'].to(device))
        yc = batch['contact_state'].numpy().astype(bool)
        c_true += yc.tolist()
        c_pred += (out['contact'].squeeze(1) > 0.5).cpu().numpy().tolist()
        i_true += batch['incline_label'].numpy()[yc].tolist()
        i_pred += out['incline'].argmax(1).cpu().numpy()[yc].tolist()
        m_true += batch['material_label'].numpy()[yc].tolist()
        m_pred += out['material'].argmax(1).cpu().numpy()[yc].tolist()
c_true, c_pred, i_true, i_pred, m_true, m_pred = map(np.array, (c_true, c_pred, i_true, i_pred, m_true, m_pred))
tp, fp, fn = (c_pred & c_true).sum(), (c_pred & ~c_true).sum(), (~c_pred & c_true).sum()
print(f"Validation contact accuracy:  {(c_true == c_pred).mean():.4f}  (precision {tp / (tp + fp):.4f}, recall {tp / (tp + fn):.4f})")
print(f"Validation incline accuracy:  {(i_true == i_pred).mean():.4f}  (contact windows only)")
print(f"Validation material accuracy: {(m_true == m_pred).mean():.4f}  (contact windows only)")

incline_names = [f"{a} deg" for a in dataset.incline_classes]
for name, t, p, labels in (("Incline", i_true, i_pred, incline_names), ("Material", m_true, m_pred, dataset.material_classes)):
    cm = pd.crosstab(pd.Series([labels[k] for k in t], name='true'), pd.Series([labels[k] for k in p], name='pred'))
    print(f"\n{name} confusion matrix (validation, contact windows)")
    print(cm.to_string())

# Export (deployment step 1)
lookup_dir = os.path.join(data_dir, 'lookups')
joint_columns = [f'leg_front_r_{j}_{q}' for q in ('position', 'velocity', 'effort') for j in (1, 2, 3)]
state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
torch.save(state, os.path.join(lookup_dir, 'implicit_classifier.pt'))
export_file = os.path.join(lookup_dir, 'implicit_classifier.npz')
np.savez(export_file,
         **{k: v.numpy() for k, v in state.items() if 'num_batches' not in k},
         channels=np.arange(9), mean=np.zeros(9, np.float32), std=np.ones(9, np.float32),
         window_size=dataset.window_size, sample_rate_hz=1000.0, joint_columns=np.array(joint_columns),
         incline_classes=np.array(dataset.incline_classes), material_classes=np.array(dataset.material_classes))
print(f"\nExported {export_file}")

Validation contact accuracy:  0.9592  (precision 0.9688, recall 0.9500)
Validation incline accuracy:  0.8575  (contact windows only)
Validation material accuracy: 0.7903  (contact windows only)

Incline confusion matrix (validation, contact windows)
pred    0 deg  15 deg  30 deg  45 deg  60 deg  75 deg
true                                                 
0 deg     752      22       0       0       4       6
15 deg     12     897      27      22      32       6
30 deg      0      76     660      93      17      35
45 deg      0      54      23     640     111      36
60 deg      4       7       1      27     884      79
75 deg      1       3       8      14      59     853

Material confusion matrix (validation, contact windows)
pred        felt pad  foam  none  rough  slip  very rough
true                                                     
felt pad         923     9    19     27    45          15
foam               7   672     6     27    59           5
none               1     0   

### ✅ Results & Reflections

By tuning various training parameters, we were able to achieve:

- **~95% accuracy** on contact detection  
- **~90% accuracy** on slope incline classification  
- **~80% accuracy** on surface material classification  

Surprisingly, these results show that joint state readings alone — without any high-end tactile sensors — can be remarkably effective for terrain understanding. This highlights the potential of proprioceptive sensing as a lightweight, low-cost alternative for real-time contact reasoning in legged robots.
